# 02 — Dataset exploration
**Challenge:** variable customer history lengths waste padding and dense attention work.

**Hypothesis:** sorting by event count reduces allocated positions. This is a CPU simulation, not a measured GPU speedup. Run from the repository root after the baseline quickstart. For a fresh Colab session use the runner notebook first.

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
ROOT = Path('runs/research_baseline')
assert (ROOT / 'raw/events.parquet').exists(), 'Run the baseline or runner first'
print(pq.read_schema(ROOT / 'raw/events.parquet'))
print(pq.read_schema(ROOT / 'raw/profiles.parquet'))

## Inspect history and event types
Why: distinguish customer profile state from temporal events before tokenization. Only inspect synthetic examples; keep raw records out of Git.

In [ ]:
events = pd.read_parquet(ROOT / 'raw/events.parquet')
profiles = pd.read_parquet(ROOT / 'raw/profiles.parquet')
lengths = events.groupby('user_id').size()
print(lengths.describe(percentiles=[.5,.9,.95,.99]))
print(events['source'].value_counts())
uid = lengths.index[0]
display(events[events.user_id == uid].sort_values('ts').head(10))
display(profiles[profiles.user_id == uid].head())

## Measure padding and attention waste
For batch lengths n_i and maximum M, allocated positions = B*M; dense attention positions = B*M². Compare random and globally sorted groups with identical data and batch size. The latter is an idealized reference, not the production sampler.

In [ ]:
%run research/experiments/02_dataset_exploration.py

In [ ]:
import json
from IPython.display import Image, display
r = json.loads(Path('research/results/02_dataset_stats.json').read_text())
display(pd.DataFrame([r['random_batching'], r['length_sorted_batching']], index=['random','sorted']))
display(Image(filename='research/results/02_sequence_distribution.png'))

## Temporal labels and limitations
Inspect label schemas for observation and prediction cutoff fields. Do not feed outcomes into input tokens. A schema alone does not establish leakage safety.

In [ ]:
for p in sorted((ROOT / 'raw/labels').glob('*.parquet')):
    print(p.name, pq.read_schema(p))

## Record conclusions
Compare event padding and attention waste, explain why fixed customer count still leads to uneven useful-token loads, and distinguish simulation from GPU throughput. Commit only the aggregate JSON, plot and reviewed notebook output. Next: token-level batching and actual kernel benchmarking.